In [1]:
import json
import random
import statistics
import sys
from datetime import datetime, timezone
from pathlib import Path

from scipy.stats import spearmanr

REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "database" / "CSCAF_public.db"
OUTPUT_FOLDER = REPO_ROOT / "results" / "output"
FIGURE_FOLDER = REPO_ROOT / "results" / "figures"
CALIBRATION = REPO_ROOT / "methodology" / "output" / "04_threshold_calibration.json"
RESAMPLES, LEVEL, HUB_MINIMUM = 2000, 0.95, 5

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert DB_PATH.exists(), "run the notebooks of data_load first"
assert CALIBRATION.exists(), "run methodology/04_threshold_calibration.ipynb first"

from cscaf.figures import RoleCoverageFigure, RoleDemandFigure, SkillsAgainstKnowledgeFigure
from cscaf.measures import AdvertisementDemand, CurriculumCoverage, KnownGroups, RoleMapping
from cscaf.storage import Storage
from cscaf.study import Study

study = Study.from_config()
SEED = study.seed
TAU = study.tau
ENDS = list(study.ends)
KEYS = study.curricula
RUN = study.run_id
SPECIFICITY = round(json.loads(CALIBRATION.read_text(encoding="utf-8"))["table_s2"]["job_advertisement"]["specificity"], 3)
DETECTION = round(1 - SPECIFICITY, 3)
storage = Storage.of(study)
assert sorted(KEYS) == sorted(storage.curricula.keys())
print("database        :", DB_PATH.relative_to(REPO_ROOT))
print("run             :", RUN)
print("tau             :", TAU, " ends:", ENDS)
print("seed            :", SEED, " resamples:", RESAMPLES)
print("specificity     :", SPECIFICITY, " detection level:", DETECTION)
print("curricula       :", len(KEYS))

database        : database\CSCAF_public.db
run             : sat-12l-sm_all-mpnet-base-v2_414c4542
tau             : 0.585  ends: [0.58, 0.595]
seed            : 42  resamples: 2000
specificity     : 0.994  detection level: 0.006
curricula       : 27


In [2]:
titles = dict(storage.advertisements.titles())
courses = {}
for key, name in storage.curricula.course_names():
    if key in KEYS:
        courses.setdefault(key, []).append(name)


known_groups = {}
for threshold in [TAU] + ENDS:
    groups = KnownGroups(study.at_tau(threshold), storage)
    ads_groups = groups.advertisements(titles)
    programme_groups = groups.curricula(courses)
    tested = sorted((e["p_value"], e["role"]) for e in programme_groups if e.get("enough"))
    holm, running = {}, 0.0
    for index, (p_value, role) in enumerate(tested):
        running = max(running, min(1.0, (len(tested) - index) * p_value))
        holm[role] = running
    for e in programme_groups:
        if e["role"] in holm:
            e["p_holm"] = holm[e["role"]]
    known_groups[f"{threshold:.3f}"] = {"advertisements": ads_groups, "curricula": programme_groups,
                                        "summary": KnownGroups.summary(ads_groups, programme_groups),
                                        "curriculum_significant_holm": sum(1 for v in holm.values() if v < 0.05)}
    print(f"tau {threshold:.3f}: {known_groups[f'{threshold:.3f}']['summary']}, "
          f"Holm {known_groups[f'{threshold:.3f}']['curriculum_significant_holm']}")

print()
for e in known_groups[f"{TAU:.3f}"]["advertisements"]:
    print(f"  {e['role']:18s} {e['advertisements']:4d}  " + (f"{e['inside']:.3f} / {e['outside']:.3f}  place "
          f"{e['rank_of_role']} ({e['strongest_role']} first)" if e.get("enough") else "not tested"))
for e in known_groups[f"{TAU:.3f}"]["curricula"]:
    print(f"  {e['role']:18s} " + (f"{e['with_course']:3d} / {e['without_course']:3d}  {e['inside']:.3f} / "
          f"{e['outside']:.3f}  p {e['p_value']:.3f}  Holm {e['p_holm']:.3f}" if e.get("enough") else "not tested"))

tau 0.585: {'advertisement_groups': 7, 'advertisement_higher': 7, 'advertisement_role_first': 6, 'curriculum_groups': 7, 'curriculum_direction': 7, 'curriculum_significant': 3, 'passes': True}, Holm 2


tau 0.580: {'advertisement_groups': 7, 'advertisement_higher': 7, 'advertisement_role_first': 6, 'curriculum_groups': 7, 'curriculum_direction': 7, 'curriculum_significant': 3, 'passes': True}, Holm 1


tau 0.595: {'advertisement_groups': 7, 'advertisement_higher': 7, 'advertisement_role_first': 6, 'curriculum_groups': 7, 'curriculum_direction': 7, 'curriculum_significant': 4, 'passes': True}, Holm 2

  PenTester           229  0.280 / 0.101  place 1 (PenTester first)
  CISO                136  0.412 / 0.248  place 1 (CISO first)
  IncidentResponder   143  0.348 / 0.179  place 1 (IncidentResponder first)
  ThreatIntel          47  0.315 / 0.102  place 1 (ThreatIntel first)
  Auditor              33  0.360 / 0.157  place 1 (Auditor first)
  RiskManager          30  0.333 / 0.178  place 1 (RiskManager first)
  Architect            87  0.346 / 0.161  place 2 (CISO first)
  Forensics             8  not tested
  Forensics           19 /   8  0.640 / 0.493  p 0.017  Holm 0.085
  PenTester           12 /  15  0.507 / 0.475  p 0.384  Holm 0.404
  IncidentResponder   13 /  14  0.688 / 0.500  p 0.006  Holm 0.040
  Auditor              6 /  21  0.510 / 0.289  p 0.006  Holm 0.040
  RiskManager   

In [3]:
mapping = RoleMapping.read(storage)
coverage = CurriculumCoverage(study, storage).statements(KEYS)
market = AdvertisementDemand(study, storage)
demand = market.statements()
role_demand_ci = market.role_interval(mapping, resamples=RESAMPLES, level=LEVEL, seed=SEED)
statements = sorted(demand["items"])
assert len(statements) == 150
role_cov = {}
for key in KEYS:
    for role, entry in CurriculumCoverage.roles(coverage[key]["items"], mapping).items():
        role_cov.setdefault(role, []).append(entry["coverage"])
names = {r: role_demand_ci[r]["role_name"] for r in role_demand_ci}
roles = {role: {"role_name": names[role], "items": role_demand_ci[role]["items"],
                "coverage_min": min(role_cov[role]), "coverage_median": statistics.median(role_cov[role]),
                "coverage_max": max(role_cov[role]), "coverage_range": max(role_cov[role]) - min(role_cov[role]),
                "demand": role_demand_ci[role]["binary_share"],
                "demand_ci": [role_demand_ci[role]["ci_low"], role_demand_ci[role]["ci_high"]]}
         for role in role_demand_ci}
for role in sorted(roles, key=lambda r: -roles[r]["demand"]):
    v = roles[role]
    print(f"{v['role_name'][:42]:42s} {v['items']:3d}  C(r) {v['coverage_min']:.3f} {v['coverage_median']:.3f} "
          f"{v['coverage_max']:.3f}  B(r) {v['demand']:.3f} [{v['demand_ci'][0]:.3f}, {v['demand_ci'][1]:.3f}]")

Chief Information Security Officer (CISO)   27  C(r) 0.111 0.704 1.000  B(r) 0.252 [0.246, 0.258]
Cybersecurity Educator                      17  C(r) 0.000 0.647 0.882  B(r) 0.205 [0.200, 0.211]
Cyber Incident Responder                    19  C(r) 0.263 0.579 0.895  B(r) 0.183 [0.179, 0.188]
Cybersecurity Risk Manager                  16  C(r) 0.188 0.562 1.000  B(r) 0.178 [0.174, 0.183]
Cybersecurity Architect                     25  C(r) 0.120 0.560 0.840  B(r) 0.164 [0.160, 0.168]
Cybersecurity Auditor                       16  C(r) 0.000 0.312 0.750  B(r) 0.158 [0.154, 0.162]
Cybersecurity Implementer                   18  C(r) 0.222 0.556 0.778  B(r) 0.153 [0.149, 0.156]
Cyber Legal, Policy & Compliance Officer    13  C(r) 0.000 0.462 0.923  B(r) 0.142 [0.138, 0.146]
Cybersecurity Researcher                    12  C(r) 0.000 0.417 0.833  B(r) 0.130 [0.126, 0.133]
Penetration Tester                          23  C(r) 0.130 0.478 0.739  B(r) 0.109 [0.107, 0.112]
Cyber Threat Intelli

In [4]:
text_of = dict(storage.framework.statement_texts())
subsection_of = {}
for sid, kind in storage.framework.statement_subsections():
    subsection_of.setdefault(kind, set()).add(sid)
skills, knowledge = sorted(subsection_of["skills"]), sorted(subsection_of["knowledge"])
shares = {s: demand["items"][s]["binary_share"] for s in statements}
taught_by = {s: sum(1 for k in KEYS if coverage[k]["items"][s]["above_tau"]) for s in statements}
untaught = [s for s in statements if taught_by[s] == 0]
per_curriculum = [{"skills": sum(1 for s in skills if coverage[k]["items"][s]["above_tau"]) / len(skills),
                   "knowledge": sum(1 for s in knowledge if coverage[k]["items"][s]["above_tau"]) / len(knowledge)}
                  for k in KEYS]
rank_agreement = spearmanr([shares[s] for s in statements], [taught_by[s] for s in statements])
communication = [s for s in sorted(statements, key=lambda s: -shares[s]) if shares[s] >= 0.20 and taught_by[s] <= 6]
split = {"skills": len(skills), "knowledge": len(knowledge),
         "skills_median": statistics.median(v["skills"] for v in per_curriculum),
         "knowledge_median": statistics.median(v["knowledge"] for v in per_curriculum),
         "knowledge_higher": sum(1 for v in per_curriculum if v["knowledge"] > v["skills"]),
         "demand_skills": statistics.mean(shares[s] for s in skills),
         "demand_knowledge": statistics.mean(shares[s] for s in knowledge)}
print(split)
print(f"Spearman of B(s) with the number of curricula covering s: {rank_agreement.statistic:.3f}")
for s in communication:
    print(f"  B {shares[s]:.3f}  covered by {taught_by[s]:2d}  {text_of[s]}")
print(f"covered by no curriculum: {len(untaught)}")
for s in untaught:
    print(f"  B {shares[s]:.4f}  {text_of[s]}")

{'skills': 83, 'knowledge': 67, 'skills_median': 0.4819277108433735, 'knowledge_median': 0.5671641791044776, 'knowledge_higher': 24, 'demand_skills': 0.16285703561911197, 'demand_knowledge': 0.11683696005951551}
Spearman of B(s) with the number of curricula covering s: 0.746
  B 0.242  covered by  6  Communicate, present and report to relevant stakeholders
  B 0.213  covered by  3  Communicate, coordinate and cooperate with internal and external stakeholders
covered by no curriculum: 5
  B 0.0062  Work under pressure
  B 0.0025  Conformity assessment standards, methodologies and frameworks
  B 0.0012  Identify and select appropriate pedagogical approaches for the intended audience
  B 0.0002  Pedagogical standards, methodologies and frameworks
  B 0.0012  Think creatively and outside the box


In [5]:
covering, decisions = {}, {}
for key, statement, sentence_id, score in storage.similarities.curriculum_best_match_rows(RUN):
    if key in KEYS and score >= TAU:
        covering.setdefault((key, sentence_id), []).append(statement)
        decisions[key] = decisions.get(key, 0) + 1
hub_ids = {sentence_id for (_, sentence_id), covered in covering.items() if len(covered) >= HUB_MINIMUM}
without_hubs = {}
for key, statement, sentence_id in storage.similarities.curriculum_matches_above(RUN, TAU):
    if key in KEYS and sentence_id not in hub_ids:
        without_hubs.setdefault(key, set()).add(statement)
hub_keys = [k for k in KEYS if decisions.get(k)]
hubness = {"hubs": len(hub_ids),
           "spearman": float(spearmanr([decisions[k] for k in hub_keys],
                                       [len(without_hubs.get(k, ())) for k in hub_keys])[0])}

covered = {k: sum(1 for s in statements if coverage[k]["items"][s]["above_tau"]) for k in KEYS}
sentences = storage.curricula.sentence_counts(RUN)
course_count = storage.curricula.course_counts()
with_sentences = spearmanr([sentences[k] for k in KEYS], [covered[k] for k in KEYS])
with_courses = spearmanr([course_count[k] for k in KEYS], [covered[k] for k in KEYS])
volume = {"sentences_rho": float(with_sentences[0]), "sentences_p": float(with_sentences[1]),
          "courses_rho": float(with_courses[0]), "courses_p": float(with_courses[1])}
print(hubness)
print(volume)

{'hubs': 74, 'spearman': 0.9570667422244197}
{'sentences_rho': 0.6319083969465649, 'sentences_p': 0.00040687057318411946, 'courses_rho': 0.3704676935896447, 'courses_p': 0.05714045790240144}


In [6]:
educator_statements = ["Identify needs in cybersecurity awareness, training and education",
                       "Design, develop and deliver learning programmes to cover cybersecurity needs"]
educator_sample = {}
for statement in educator_statements:
    rows = storage.similarities.advertisement_sentences_for_statement(statement, TAU)
    random.seed(SEED)
    educator_sample[statement] = {"advertisements": len(rows),
                                  "sample": [sentence for (sentence,) in random.sample(rows, 10)]}
    print(f"{statement}: {len(rows)} advertisements")

Identify needs in cybersecurity awareness, training and education: 2304 advertisements


Design, develop and deliver learning programmes to cover cybersecurity needs: 2036 advertisements


In [7]:
FIGURE_FOLDER.mkdir(parents=True, exist_ok=True)
figures = [RoleCoverageFigure(role_cov, names).draw(FIGURE_FOLDER / "figure_3_role_coverage.png"),
           RoleDemandFigure(role_demand_ci, names).draw(FIGURE_FOLDER / "figure_4_role_demand.png"),
           SkillsAgainstKnowledgeFigure(per_curriculum, len(skills), len(knowledge)).draw(
               FIGURE_FOLDER / "figure_5_skills_against_knowledge.png")]
for path in figures:
    print("written to", path.relative_to(REPO_ROOT))

written to results\figures\figure_3_role_coverage.png
written to results\figures\figure_4_role_demand.png
written to results\figures\figure_5_skills_against_knowledge.png


In [8]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
output = OUTPUT_FOLDER / "05_role_level_results.json"
record = {
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "paper_section": "4.1",
    "run": RUN, "tau": TAU, "ends": ENDS, "seed": SEED, "resamples": RESAMPLES, "level": LEVEL,
    "specificity_advertisements": SPECIFICITY, "detection_level": DETECTION,
    "curricula": KEYS, "advertisements": demand["n_ads"], "statements": len(statements),
    "known_groups": known_groups,
    "roles": roles,
    "skills_against_knowledge": split,
    "rank_agreement": {"rho": float(rank_agreement.statistic), "p": float(rank_agreement.pvalue)},
    "communication": [{"statement": text_of[s], "demand": shares[s], "covered_by": taught_by[s]} for s in communication],
    "untaught": [{"statement": text_of[s], "demand": shares[s]} for s in untaught],
    "hubness": hubness,
    "text_volume": volume,
    "educator_sample": educator_sample,
    "known_group_keywords": {key: study.section("known_groups")[key] for key in ("advertisement_titles", "course_names")},
    "figures": [str(path.relative_to(REPO_ROOT)) for path in figures],
}
output.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
print("written to", output.relative_to(REPO_ROOT))

written to results\output\05_role_level_results.json
